In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch


/home/lang-chain/Documents/Astra_agentic_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    print('cuda')

cuda


In [2]:
torch.cuda.empty_cache()

In [3]:
model_path = "./my_4bit_model"  
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
non_quanitzed_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True  
)

Loading weights: 100%|██████████| 355/355 [00:15<00:00, 22.43it/s]


In [7]:
prompt = "Language modeling is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

# Decode and print the response
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(response)


Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
Loading weights: 100%|██████████| 355/355 [00:17<00:00, 20.75it/s]


Language modeling is one of the key components of most deep learning architectures (BIBREF14). There are various types of sequence models that are able to predict the next word given a context word or phrase. These models are based on neural network architectures such as long short-term memory (LSTM) (BIBREF29) and gated recurrent units (GRU) (BIBREF30). The choice of neural model architecture depends on the use case of the sequence modeling: for text applications, a LSTM is the


In [4]:
# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


In [7]:
bnb_config

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="cuda",
    trust_remote_code=True,
    dtype=torch.float16
)


Loading weights: 100%|██████████| 355/355 [00:11<00:00, 30.52it/s]


In [6]:
memory_bytes = model.get_memory_footprint()
memory_gb = memory_bytes / (1024 ** 3)
print(f"Model memory footprint: {memory_gb:.2f} GB")

Model memory footprint: 4.55 GB


In [7]:
model.save_pretrained("./my_4bit_model")

Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.16s/it]


In [9]:
import torch
import psutil
import os

def get_model_size(model):
    param_count = sum(p.numel() for p in model.parameters())
    param_size_bytes = param_count * 2 
    
    buffer_size = 0
    if hasattr(model, 'model') and hasattr(model.model, 'buffers'):
        buffer_size = sum(b.numel() for b in model.model.buffers()) * 2
    
    total_size_gb = (param_size_bytes + buffer_size) / (1024**3)
    
    return {
        "parameters": f"{param_count:,}",
        "parameters_billions": param_count / 1e9,
        "estimated_size_gb": total_size_gb
    }

size_info = get_model_size(model)
print(f"Model parameters: {size_info['parameters']}")
print(f"Model size (billions): {size_info['parameters_billions']:.2f}B")
print(f"Estimated model size: {size_info['estimated_size_gb']:.2f} GB")

Model parameters: 4,060,614,656
Model size (billions): 4.06B
Estimated model size: 7.56 GB


In [11]:

prompt = "What is a checking account?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(response)


What is a checking account? What is a savings account? What is the difference between the two? These are questions that most of us have probably asked at some point or another.

The truth is, the difference between a checking account and a savings account is fairly simple. The major difference between the two is the way that each account is used and the way that money is withdrawn from the account. However, there are some other differences that you should be aware of before you decide on a checking or savings account.

What is a Checking Account?

A checking account is a deposit account held at a bank or other financial institution. This account allows you to write checks or use a debit card to make purchases. It is important to note that checking accounts do not usually earn any interest.

This account is designed to be a way to pay for items that you need to purchase. Therefore, it is often the account that is used to pay for things like bills or groceries. In order to make payments,

In [29]:
user_input = "Checking Account भएको नेपाली भाषामा, चेकिङ खातालाई सामान्यतया करेन्ट एकाउन्ट वा साधारण रूपमा खाता भनिन्छ। यद्यपि अङ्ग्रेजी शब्द ‘चेकिङ एकाउन्ट’ प्रायः बैंकिङ सन्दर्भमा प्रयोग गरिन्छ, ‘एकाउन्ट’ को प्रत्यक्ष अनुवाद खाता हो, र बारम्बार जम्मा तथा निकासीको लागि लेनदेन खाताको अवधारणा यसै शब्द मार्फत बुझिन्छ।"

context = "चेकिङ खातालाई सामान्यतया करेन्ट एकाउन्ट वा साधारण रूपमा खाता भनिन्छ। यद्यपि अङ्ग्रेजी शब्द 'चेकिङ एकाउन्ट' प्रायः बैंकिङ सन्दर्भमा प्रयोग गरिन्छ, 'एकाउन्ट' को प्रत्यक्ष अनुवाद खाता हो, र बारम्बार जम्मा तथा निकासीको लागि लेनदेन खाताको अवधारणा यसै शब्द मार्फत बुझिन्छ।"

# First, detect if the user is asking a relevant question
prompt = f"""### System Instructions:
You are a helpful Nepali banking assistant. Your task:

1. If the user's question is **unclear, nonsensical, or contains unrelated content** (like family members, princes, votes), politely ask them to rephrase their question about banking.

2. If the question is **relevant to banking**, answer based ONLY on the context provided.

3. Respond in **Romanized Nepali** (Nepali written with English/Latin letters) to match the user's writing style.

4. Be concise and helpful.

### Context (Banking Information in Nepali):
{context}

### User Question (Romanized Nepali):
{user_input}

### Analysis:
First, determine if this question is about banking or unrelated.

### Response:
"""

# Generation with appropriate settings
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,  # Enable sampling for more natural responses
    temperature=0.3,  # Low temperature for focused responses
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract just the response part
if "### Response:" in response:
    final_output = response.split("### Response:")[-1].strip()
else:
    final_output = response.strip()

print(final_output)

चिनाई खत भेटिने राम्रो विचार हो भरिएका छैनन्, तर आफ्नो खर्च लगाउँछौं भिनेको कुरा आर्थिक रुपमै समाधिन गाह्रो हो।


In [28]:
user_input = "Checking Account भएको छैन भाइ र बहिनी यस्ता गर्ने अवसर छ भतीदार राजकुमारको साथ लाएका छन् तर त्यसको प्रतिकूल अनुभव भिन्न भोट भूमि"

context = "चेकिङ खातालाई सामान्यतया करेन्ट एकाउन्ट वा साधारण रूपमा खाता भनिन्छ। यद्यपि अङ्ग्रेजी शब्द 'चेकिङ एकाउन्ट' प्रायः बैंकिङ सन्दर्भमा प्रयोग गरिन्छ, 'एकाउन्ट' को प्रत्यक्ष अनुवाद खाता हो, र बारम्बार जम्मा तथा निकासीको लागि लेनदेन खाताको अवधारणा यसै शब्द मार्फत बुझिन्छ।"

# First, detect if the user is asking a relevant question
prompt = f"""### System Instructions:
You are a helpful Nepali banking assistant. Your task:

1. If the user's question is **unclear, nonsensical, or contains unrelated content**

2. If the question is **relevant to banking**, answer based ONLY on the context provided.

3. Respond in **Romanized Nepali** (Nepali written with English/Latin letters) to match the user's writing style.

4. Be concise and helpful.

### Context (Banking Information in Nepali):
{context}

### User Question (Romanized Nepali):
{user_input}

### Analysis:
First, determine if this question is about banking or unrelated.

### Response:
"""

# Generation with appropriate settings
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to('cpu')

outputs = non_quanitzed_model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,  # Enable sampling for more natural responses
    temperature=0.3,  # Low temperature for focused responses
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "### Response:" in response:
    final_output = response.split("### Response:")[-1].strip()
else:
    final_output = response.strip()

print(final_output)

यस ब्याख्या लिने काम भर्खर रहेको हो भेटिन र साहिब यो बेला आफ्नो चेनिङ कारोबाडी रख्नु अर्को काहाँ गारी छ ।


In [31]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",         
    trust_remote_code=True,
   
)

Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
Loading weights: 100%|██████████| 355/355 [03:48<00:00,  1.55it/s]


In [ ]:
def bank_query(user_input: str, context: str = "") -> str:
    """
    Correct OLMo-2 inference using manual instruct format.
    Works for English, Nepali, and Romanized Nepali.
    """
    system_prompt = """You are a helpful Nepali banking assistant.
Answer only from the provided context.
If context is missing, redirect to branch.
Reply in the same script as the question."""

    user_content = f"Context:\n{context}\n\nQuestion: {user_input}" \
                   if context else user_input

    # Manual prompt construction (OLMo-2 instruct format)
    prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_content}\n<|assistant|>\n"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
context = """
Bachat khata kholna minimum Rs. 100 chaincha.
Byaj dar barshik 5% cha.
Nagarikta praman patra ra 2 wata photo aawashyak cha.
"""

print(bank_query("bachat khata kholna k chaincha?", context))

# Test 2: Devanagari
print(bank_query("बचत खाता खोल्न के चाहिन्छ?", context))

# Test 3: Out of scope
print(bank_query("aaja mausam kasto cha?"))

# Test 4: Sensitive
print(bank_query("mero balance kati cha?"))

In [1]:
import tiny_llm_scratch_with_tokenizer as m
print(m.__file__)          # confirm which .so is loaded

/home/lang-chain/Documents/tiny_LLM_scratch_with_tokenizer/.venv/lib/python3.11/site-packages/tiny_llm_scratch_with_tokenizer/__init__.py


In [2]:
import sys
import statistics
import time
from tiny_llm_scratch_with_tokenizer import PyNepBPETokenizer

VOCAB_TSV = 'vocab_nepbpe/nepbpe_vocab_bilingual_new.tsv'
#"dataset_ne/nepbpe_vocab_new.tsv"

In [7]:
FOLDING_RULES = [
    ("सङ्ग", "संग"),
    ("सँग", "संग"),
]

data="""
विद्यालयमा
विद्यालयको
विद्यालयदेखि
विद्यालयसम्म
विद्यालयबाट
"""
tok = PyNepBPETokenizer(folding_rules=FOLDING_RULES)

# ----- ADD THIS LINE -----
tok.load_vocab_tsv(VOCAB_TSV)
# --------------------------

print("id:", tok.vocab_get_id(data))  
print("size:", tok.vocab_size())           

ids = tok.encode(data)
print("surfaces:", [tok.get_token_surface(i) for i in ids])


id: None
size: 48001
surfaces: ['▁', 'Ċ', 'विद्यालय', 'मा', 'Ċ', 'विद्यालय', 'को', 'Ċ', 'विद्यालय', 'देखि', 'Ċ', 'विद्यालय', 'सम्म', 'Ċ', 'विद्यालय', 'बाट', 'Ċ']


In [1]:
import tiny_llm_scratch_with_tokenizer as m
print(m.__file__)          

/home/lang-chain/Documents/tiny_LLM_scratch_with_tokenizer/.venv/lib/python3.11/site-packages/tiny_llm_scratch_with_tokenizer/__init__.py


In [ ]:
for word in data.split():
    ids = tok.encode(word)
    print(word, "->", [tok.get_token_surface(i) for i in ids])
    

विद्यालयमा -> ['▁विद्यालयमा']
विद्यालयको -> ['▁विद्यालयको']
विद्यालयदेखि -> ['▁विद्यालय', 'देखि']
विद्यालयसम्म -> ['▁विद्यालय', 'सम्म']
विद्यालयबाट -> ['▁विद्यालयबाट']


In [ ]:
ids = tok.encode("सन् 2020 मा गा.वि.स.को निर्णय")
surfaces = [tok.get_token_surface(i) for i in ids]

surfaces
['▁', 'सन्', '▁', '2020', '▁', 'मा', '▁', 'गा.वि.स.', 'को', '▁निर्णय']

['▁', 'सन्', '▁', '2020', '▁', 'मा', '▁', 'गा.वि.स.', 'को', '▁निर्णय']